In [7]:
import sys
import os

# SPARK_HOME path
os.environ["SPARK_HOME"] = "/home/fragkiska/spark"

# Add pyspark to Python path
sys.path.append("/home/fragkiska/spark/python")
sys.path.append("/home/fragkiska/spark/python/lib/py4j-0.10.9.7-src.zip")


import pyspark
from pyspark.sql import SparkSession


In [ ]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, udf, count, desc, rank, sum as _sum
from pyspark.sql.types import StringType
from pyspark.sql.window import Window


import time

# ---------------------------------------------------------------
# Spark Session with required executor configuration
# ---------------------------------------------------------------
spark = SparkSession.builder \
    .appName("Query1-AdvancedDBs") \
    .config("spark.executor.instances", "4") \
    .config("spark.executor.cores", "1") \
    .config("spark.executor.memory", "2g") \
    .config("spark.hadoop.fs.s3a.access.key", "YOUR_KEY") \
    .config("spark.hadoop.fs.s3a.secret.key", "YOUR_SECRET") \
    .config("spark.hadoop.fs.s3a.endpoint", "s3.eu-central-1.amazonaws.com") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()


In [9]:
from pathlib import Path

project_root = Path.cwd()

data_dir = project_root / "data"

crime_data_2010_2019 = data_dir / "LA_Crime_Data_2010_2019.csv"
crime_data_2020_2025 = data_dir / "LA_Crime_Data_2020_2025.csv"

print(crime_data_2010_2019)
print(crime_data_2020_2025)

df1 = spark.read.csv(str(crime_data_2010_2019), header=True, inferSchema=True)
df2 = spark.read.csv(str(crime_data_2020_2025), header=True, inferSchema=True)

crime_data = df1.unionByName(df2)

crime_data.show(5)   

/mnt/c/Users/user/Desktop/advanced_dbs/Advanced-Databases-Project/data/LA_Crime_Data_2010_2019.csv
/mnt/c/Users/user/Desktop/advanced_dbs/Advanced-Databases-Project/data/LA_Crime_Data_2020_2025.csv


+---------+--------------------+--------------------+--------+----+---------+-----------+--------+------+--------------------+--------------+--------+--------+------------+---------+--------------------+--------------+--------------------+------+------------+--------+--------+--------+--------+--------------------+--------------------+-------+---------+
|    DR_NO|           Date Rptd|            DATE OCC|TIME OCC|AREA|AREA NAME|Rpt Dist No|Part 1-2|Crm Cd|         Crm Cd Desc|       Mocodes|Vict Age|Vict Sex|Vict Descent|Premis Cd|         Premis Desc|Weapon Used Cd|         Weapon Desc|Status| Status Desc|Crm Cd 1|Crm Cd 2|Crm Cd 3|Crm Cd 4|            LOCATION|        Cross Street|    LAT|      LON|
+---------+--------------------+--------------------+--------+----+---------+-----------+--------+------+--------------------+--------------+--------+--------+------------+---------+--------------------+--------------+--------------------+------+------------+--------+--------+--------+--

In [ ]:
# DataFrame Implementation without UDF
# Start timing
start_time_df = time.time()

from pyspark.sql.functions import col, when

# Φιλτράρισμα για aggravated assault
agg_assault = crime_data.filter(
    col("Crm Cd Desc").contains("AGGRAVATED ASSAULT")
)

# Κατηγοριοποίηση ηλικιών χωρίς UDF (μόνο withColumn + when)
categorized = agg_assault.withColumn(
    "Age_Group",
    when(col("Vict Age") < 18, "Παιδιά")
    .when((col("Vict Age") >= 18) & (col("Vict Age") <= 24), "Νεαροί ενήλικες")
    .when((col("Vict Age") >= 25) & (col("Vict Age") <= 64), "Ενήλικοι")
    .when(col("Vict Age") > 64, "Ηλικιωμένοι")
    .otherwise("Άγνωστο")
)

# Ομαδοποίηση και καταμέτρηση
result = (
    categorized.groupBy("Age_Group")
    .count()
    .orderBy(col("count").desc())
)

result.show(truncate=False)


# End timing
end_time_df = time.time()

# Calculate elapsed time
elapsed_time_df = end_time_df - start_time_df
print(f"DataFrame without UDF Implementation Time: {elapsed_time_df:.2f} seconds")

+---------------+------+
|Age_Group      |count |
+---------------+------+
|Ενήλικοι       |121660|
|Νεαροί ενήλικες|33758 |
|Παιδιά         |16014 |
|Ηλικιωμένοι    |6011  |
+---------------+------+

DataFrame without UDF Implementation Time: 4.20 seconds


In [11]:
# DataFrame Implementation with UDF
# Start timing
start_time_df = time.time()

def age_category(age):
    if age is None:
        return "Άγνωστο"
    try:
        age = int(age)
    except:
        return "Άγνωστο"

    if age < 18:
        return "Παιδιά"
    elif 18 <= age <= 24:
        return "Νεαροί ενήλικες"
    elif 25 <= age <= 64:
        return "Ενήλικοι"
    elif age > 64:
        return "Ηλικιωμένοι"
    else:
        return "Άγνωστο"

from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

age_category_udf = udf(age_category, StringType())


agg_assault = crime_data.filter(
    col("Crm Cd Desc").contains("AGGRAVATED ASSAULT")
)

categorized_udf = agg_assault.withColumn(
    "Age_Group",
    age_category_udf(col("Vict Age"))
)


result_udf = (
    categorized_udf.groupBy("Age_Group")
    .count()
    .orderBy(col("count").desc())
)

result_udf.show(truncate=False)


# End timing
end_time_df = time.time()

# Calculate elapsed time
elapsed_time_df = end_time_df - start_time_df
print(f"DataFrame with UDF Implementation Time: {elapsed_time_df:.2f} seconds")

+---------------+------+
|Age_Group      |count |
+---------------+------+
|Ενήλικοι       |121660|
|Νεαροί ενήλικες|33758 |
|Παιδιά         |16014 |
|Ηλικιωμένοι    |6011  |
+---------------+------+

DataFrame with UDF Implementation Time: 4.97 seconds


In [ ]:
import time

# ---------------------------------------------------------
# RDD Implementation
# ---------------------------------------------------------

start_time_rdd = time.time()

# Convert DataFrame to RDD
rdd = crime_data.rdd

# Filter for aggravated assault
assault_rdd = rdd.filter(
    lambda row: row["Crm Cd Desc"] is not None 
    and "AGGRAVATED ASSAULT" in row["Crm Cd Desc"].upper()
)

# Map → (age group, 1)
age_group_rdd = assault_rdd.map(
    lambda row: (
        "Παιδιά" if 0 < row["Vict Age"] < 18 else
        "Νεαροί Ενήλικες" if 18 <= row["Vict Age"] <= 24 else
        "Ενήλικες" if 25 <= row["Vict Age"] <= 64 else
        "Ηλικιωμένοι" if row["Vict Age"] > 64 else
        "Άγνωστο",
        1
    )
)

# Reduce
age_group_counts_rdd = age_group_rdd.reduceByKey(lambda x, y: x + y)

# Sort descending
sorted_age_group_counts_rdd = age_group_counts_rdd.sortBy(lambda x: x[1], ascending=False)

# Collect
results = sorted_age_group_counts_rdd.collect()

# Print results
for group, count in results:
    print(f"{group}: {count}")

end_time_rdd = time.time()
print(f"RDD Implementation Time: {end_time_rdd - start_time_rdd:.2f} seconds")

Ενήλικες: 121660
Νεαροί Ενήλικες: 33758
Παιδιά: 10904
Ηλικιωμένοι: 6011
Άγνωστο: 5110
RDD Implementation Time: 27.37 seconds
